In [ ]:
"""
BMRI Agentic Pipeline (Gemini + Deterministic Analytics)

This script implements an evidence-backed, reproducible pipeline for:
- Parsing SEC 10-K Item 1 text files (already extracted to .txt)
- Building a Business Model Canvas Summary (BMCS) per firm-year via LLM
- Rating PwC Business Model Reinvention (BMRI) types per firm-year via LLM
- Computing deterministic year-to-year transitions (presence/onset/offset/pivots)
- Computing "material change" as a *within-firm* relative (percentile-ranked) BMCS change

IMPORTANT (ground-truth constraint):
All BMRI type classification decisions MUST be based only on:
1) Item 1 text excerpts provided (business description evidence)
2) PwC's definition of business model reinvention (general concept), provided as a local document
3) PwC’s definitions and examples for each reinvention type, provided per type (local document)

No web sources are used by this pipeline.

Google GenAI SDK:
- Uses `google-genai` (official) and structured outputs (JSON schema).
"""

from __future__ import annotations

import dataclasses
import traceback
import hashlib
import json
import os
import re
import shutil
import json_repair
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Sequence, Tuple
import tempfile
import threading
import random
from contextlib import contextmanager, nullcontext
from collections import Counter
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel, Field, ValidationError, field_validator, conlist


# Google GenAI SDK (official)
from google import genai

try:
    from dotenv import load_dotenv
except Exception:  # pragma: no cover
    load_dotenv = None

In [ ]:
CSV_SEP = ";"
CSV_ENCODING = "utf-8-sig"

def save_csv_excel(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        path,
        index=False,
        sep=CSV_SEP,
        encoding=CSV_ENCODING,
        lineterminator="\n",
    )

def save_xlsx(df: pd.DataFrame, path: Path, sheet_name: str = "Sheet1") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

def write_outputs_for_company(company_dir: Path, fname_base: str, df: pd.DataFrame) -> Path:
    """
    Write Excel-only output. Returns path to XLSX written.
    """
    xlsx_path = company_dir / f"{fname_base}.xlsx"
    # Excel sheet names max 31 chars
    sheet = fname_base[:31]
    save_xlsx(df, xlsx_path, sheet_name=sheet)
    return xlsx_path

def _clean_json_text(text: str) -> str:
    t = (text or "").strip()

    # Strip ```json ... ``` fences if model returns them
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*", "", t)
        t = re.sub(r"\s*```$", "", t).strip()

    # Best-effort: extract first JSON object
    start = t.find("{")
    end = t.rfind("}")
    if start != -1 and end != -1 and end > start:
        t = t[start : end + 1].strip()

    return t


In [ ]:
# -----------------------------
# Canonical labels & constants
# -----------------------------

PWC_TYPE_IDS: Dict[str, str] = {
    "XaaS": "xaaS",
    "Digital products": "digital_products",
    "Physical/connected products": "physical_connected_products",
    "Ecosystem/platform play": "ecosystem_platform",
    "Channel innovation": "channel_innovation",
}

CANONICAL_INITIATIVE_LABELS: List[str] = [
    "Cloud",
    "Subscription/Recurring",
    "Marketplace/Platform",
    "Direct-to-Consumer",
    "Data/AI",
    "Connected devices",
    "Payments",
    "Healthcare services",
    "SUPPLY_CHAIN_LOGISTICS",
    "ELECTRIFICATION_MOBILITY", 
    "OMNICHANNEL_RETAIL", 
    "RESTRUCTURING_EFFICIENCY",
    "SECURITY_PRIVACY",
    "Other",  # only if necessary
]

EXIT_CUE_PATTERNS = [
    r"\bdivest(?:ed|iture|ing)?\b",
    r"\bexit(?:ed|ing)?\b",
    r"\bdiscontinu(?:e|ed|ing)\b",
    r"\bno longer\b",
    r"\bceas(?:e|ed|ing)\b",
    r"\bsold\b",
    r"\bshut(?:\s+down)?\b",
]

# BMCS fields for evidence enforcement
ALL_BMCS_FIELDS = ("customer_segments","offer","channels","customer_relationships","revenue_model","key_resources","key_activities","key_partners","cost_structure")

# Evidence enforcement: default to all 14 BMCS fields (can be narrowed in PipelineConfig)
DEFAULT_EVIDENCE_FIELDS = ALL_BMCS_FIELDS

In [ ]:
# -----------------------------
# Utility: robust RTF -> text
# -----------------------------

def rtf_to_text(rtf: str) -> str:
    """
    Best-effort RTF->text converter.
    Good enough for extracting PwC definitions/examples embedded in a simple .rtf.
    """
    # Convert unicode escapes \uNNNN?
    def uni(m: re.Match) -> str:
        val = int(m.group(1))
        if val < 0:
            val += 65536
        return chr(val)

    rtf = re.sub(r"\\u(-?\d+)\??", uni, rtf)

    # Remove font/color/list tables broadly (nested groups are hard; we'll also clean post-hoc)
    rtf = re.sub(r"{\\fonttbl.*?}", "", rtf, flags=re.S)
    rtf = re.sub(r"{\\colortbl.*?}", "", rtf, flags=re.S)
    rtf = re.sub(r"{\\\*\\expandedcolortbl.*?}", "", rtf, flags=re.S)
    rtf = re.sub(r"{\\\*\\listtable.*?}", "", rtf, flags=re.S)
    rtf = re.sub(r"{\\\*\\listoverridetable.*?}", "", rtf, flags=re.S)

    # Paragraph/line markers (word boundary so \pard doesn't become "\n" + "d")
    rtf = re.sub(r"\\par\b", "\n", rtf)
    rtf = re.sub(r"\\line\b", "\n", rtf)
    rtf = re.sub(r"\\tab\b", "\t", rtf)

    # Remove hex escapes \'hh
    rtf = re.sub(r"\\'[0-9a-fA-F]{2}", "", rtf)

    # Remove control words
    rtf = re.sub(r"\\\*\\[a-zA-Z]+\d* ?", "", rtf)  # \*\something
    rtf = re.sub(r"\\[a-zA-Z]+\d* ?", "", rtf)      # \something

    # Remove braces
    rtf = rtf.replace("{", "").replace("}", "")

    # Cleanup common leftover artifacts
    out_lines: List[str] = []
    for line in rtf.splitlines():
        s = line.strip()
        if not s:
            out_lines.append("")
            continue

        # Drop obvious font names / leftover list marker commands
        if s.endswith(";") and len(s) <= 40:
            continue
        if s.startswith("\\disc") or s.startswith("\\circle"):
            continue
        if re.fullmatch(r"[;0-9\-\s]+", s):
            continue

        # Remove weird list residuals
        s = re.sub(r";;?-?\d+;?", "", s).strip()
        s = s.replace("\\disc", "").replace("\\circle", "").strip()
        if not s:
            continue
        out_lines.append(s)

    out = "\n".join(out_lines)
    out = re.sub(r"\n{3,}", "\n\n", out)
    out = re.sub(r"[ \t]{2,}", " ", out)
    return out.strip()


In [ ]:
def parse_pwc_definitions_from_rtf(rtf_path: Path) -> Dict[str, str]:
    """
    Parses a PwC type-definition RTF into:
    - 'general' : the general BMRI concept paragraph(s)
    - one entry per type_id in PWC_TYPE_IDS values
    """
    raw = rtf_path.read_text(errors="ignore")
    txt = rtf_to_text(raw)

    # Best-effort split by numbered headings like "1) XaaS ..." etc.
    parts = re.split(r"(?m)^\s*(\d\))\s*", txt)
    # If split fails, fall back to whole text
    if len(parts) < 3:
        return {"general": txt}

    # parts structure: [pre, "1)", rest1, "2)", rest2, ...]
    preamble = parts[0].strip()

    type_blocks: Dict[str, str] = {"general": preamble}
    for i in range(1, len(parts) - 1, 2):
        heading_num = parts[i].strip()   # e.g., "1)"
        body = parts[i + 1].strip()

        # Identify which type this block refers to using the known names
        normalized = body.lower()
        if "xaas" in normalized or "as-a-service" in normalized or "as‑a‑service" in normalized:
            type_blocks[PWC_TYPE_IDS["XaaS"]] = body
        elif "digital product" in normalized:
            type_blocks[PWC_TYPE_IDS["Digital products"]] = body
        elif "physical" in normalized or "connected product" in normalized:
            type_blocks[PWC_TYPE_IDS["Physical/connected products"]] = body
        elif "ecosystem" in normalized or "platform" in normalized:
            type_blocks[PWC_TYPE_IDS["Ecosystem/platform play"]] = body
        elif "channel" in normalized or "disintermed" in normalized:
            type_blocks[PWC_TYPE_IDS["Channel innovation"]] = body
        else:
            # unknown bucket; keep for debugging
            type_blocks[f"unknown_block_{heading_num}"] = body

    return type_blocks 

def parse_ticker_company_pairs_from_rtf(rtf_path: Path) -> Dict[str, str]:
    raw = rtf_path.read_text(errors="ignore")
    txt = rtf_to_text(raw)
    lines = [l.strip() for l in txt.splitlines() if l.strip()]

    # Expect header lines that include "Ticker" and "Company_Name"
    # We'll pair ticker + company name afterwards.
    cleaned = [l for l in lines if l.lower() not in ("ticker", "company_name")]
    pairs: Dict[str, str] = {}
    i = 0
    while i + 1 < len(cleaned):
        ticker = cleaned[i].strip().upper()
        name = cleaned[i + 1].strip()
        if re.fullmatch(r"[A-Z]{1,6}", ticker):
            pairs[ticker] = name
            i += 2
        else:
            i += 1
    return pairs

In [ ]:
# -----------------------------
# Item1 file parsing & cleaning
# -----------------------------

@dataclass(frozen=True)
class FileMeta:
    sector: Optional[str]
    ticker: str
    year: int
    company: Optional[str] = None


def parse_item1_filename(path: Path) -> Optional[FileMeta]:
    """
    Robust filename parser.

    Handles:
    - Sector_Company_TICKER_Year.txt
    - Sector_TICKER_Year.txt
    - ... "updated" variants, extra ".txt", spacing issues

    Returns None if year/ticker cannot be inferred.
    """
    name = path.name

    # Normalize
    name = re.sub(r"\s+", " ", name).strip()

    # Remove extension(s) like .txt, .TXT, and weird "...updated txt"
    lowered = name.lower()
    lowered = lowered.replace(".txt", "")
    lowered = lowered.replace(".text", "")
    lowered = re.sub(r"\bupdated\b", "", lowered)
    lowered = re.sub(r"\s+", " ", lowered).strip()

    # Replace separators to uniform underscores
    uniform = re.sub(r"[ \-]+", "_", lowered)
    uniform = re.sub(r"_+", "_", uniform).strip("_")

    # Find last year occurrence
    years = re.findall(r"(19|20)\d{2}", uniform)
    if not years:
        return None
    year = int(years[-1])

    # Find ticker close to year: pattern TICKER_YEAR
    m = None
    for mm in re.finditer(r"([a-z]{1,6})_((19|20)\d{2})$", uniform):
        m = mm
    if m is None:
        # fallback: find "..._TICKER_YYYY..." anywhere, use last
        for mm in re.finditer(r"([a-z]{1,6})_((19|20)\d{2})", uniform):
            m = mm
        if m is None:
            return None

    ticker = m.group(1).upper()
    year = int(m.group(2))

    prefix = uniform[: m.start()].strip("_")
    tokens = prefix.split("_") if prefix else []

    sector = tokens[0].title() if tokens else None
    company = None
    if len(tokens) >= 2:
        # If there's a company field, keep it best-effort
        company = " ".join(t.title() for t in tokens[1:])

    return FileMeta(sector=sector, ticker=ticker, year=year, company=company)


In [ ]:
def clean_text(raw: str) -> str:
    """
    Heuristic cleanup for messy SEC text conversions.
    Removes common headers/footers/page numbers and stitches broken lines.
    """
    s = raw.replace("\r\n", "\n").replace("\r", "\n")

    # Remove null bytes
    s = s.replace("\x00", "")

    # Join hyphenated line breaks: "transfor-\nmation" -> "transformation"
    s = re.sub(r"(\w)-\n(\w)", r"\1\2", s)

    lines = s.splitlines()

    # Compute line frequencies to remove repeating headers/footers
    norm_lines = [re.sub(r"\s+", " ", ln).strip().lower() for ln in lines]
    freq: Dict[str, int] = {}
    for ln in norm_lines:
        if not ln:
            continue
        freq[ln] = freq.get(ln, 0) + 1

    cleaned_lines: List[str] = []
    for ln, nln in zip(lines, norm_lines):
        orig = ln.strip()

        # Drop empty lines (keep as blank marker)
        if not orig:
            cleaned_lines.append("")
            continue

        # Page markers / standalone numbers
        if re.fullmatch(r"\d{1,4}", orig):
            continue
        if re.match(r"^\s*page\s+\d+(\s+of\s+\d+)?\s*$", orig, flags=re.I):
            continue

        # Table-of-contents / dotted leaders
        if "table of contents" in nln:
            continue
        if re.fullmatch(r"\.*\s*\d+\s*", orig):
            continue

        # Repeating header/footer (short-ish lines repeated a lot)
        if freq.get(nln, 0) >= 10 and len(nln) <= 80:
            continue

        # All-caps section dividers (very short) - usually noise
        if len(orig) <= 5 and orig.isupper():
            continue

        # Lines that are mostly punctuation
        if len(re.sub(r"[\W_]+", "", orig)) == 0:
            continue

        cleaned_lines.append(orig)

    s2 = "\n".join(cleaned_lines)

    # Collapse excessive whitespace
    s2 = re.sub(r"[ \t]{2,}", " ", s2)
    s2 = re.sub(r"\n{3,}", "\n\n", s2)

    return s2.strip()


In [ ]:
@dataclass
class Paragraph:
    para_id: str
    text: str


def split_into_paragraphs(cleaned: str, min_chars: int = 220) -> List[Paragraph]:
    """
    Splits cleaned text into paragraphs using blank-line boundaries,
    then merges very short fragments to reduce "broken sentence" issues.
    """
    blocks = re.split(r"\n\s*\n+", cleaned)
    blocks = [re.sub(r"\s+", " ", b).strip() for b in blocks]
    blocks = [b for b in blocks if b]

    merged: List[str] = []
    i = 0
    while i < len(blocks):
        b = blocks[i]
        # If too short, merge with next if possible
        if len(b) < min_chars and i + 1 < len(blocks):
            merged.append((b + " " + blocks[i + 1]).strip())
            i += 2
        else:
            merged.append(b)
            i += 1

    paras: List[Paragraph] = []
    for idx, txt in enumerate(merged, start=1):
        para_id = f"p{idx:04d}"
        paras.append(Paragraph(para_id=para_id, text=txt))
    return paras


@dataclass
class DocumentYear:
    ticker: str
    year: int
    sector: Optional[str]
    company: Optional[str]
    path: Path
    paragraphs: List[Paragraph]

    @property
    def para_id_set(self) -> set[str]:
        return {p.para_id for p in self.paragraphs}



In [ ]:
# -----------------------------
# Stage 1: Retrieval index (TF-IDF, per doc-year)
# -----------------------------

class DocTfidfIndex:
    """
    Stage 1 retrieval index for a single doc-year.

    NOTE: This is NOT an "embeddings + vector DB" index.
    It's a per-doc TF-IDF matrix used for local retrieval (RAG-style),
    primarily to keep prompts short and evidence-grounded.
    """

    def __init__(self, paragraphs: Sequence[Paragraph]):
        self.paragraphs = list(paragraphs)
        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            token_pattern=r"(?u)\b\w+\b",
        )
        corpus = [p.text for p in self.paragraphs]
        self.matrix = self.vectorizer.fit_transform(corpus)  # shape (n_paras, n_terms)

    def top_k(self, query: str, k: int = 8) -> List[Tuple[str, str, float]]:
        qv = self.vectorizer.transform([query])
        sims = cosine_similarity(qv, self.matrix)[0]
        idxs = np.argsort(-sims)[:k]
        out: List[Tuple[str, str, float]] = []
        for i in idxs:
            p = self.paragraphs[int(i)]
            out.append((p.para_id, p.text, float(sims[int(i)])))
        return out


In [ ]:
# -----------------------------
# Retrieval upgrades + Selector Agent (NEW)
# -----------------------------

STOP_PHRASES = {
    "Item 1", "Item 1A", "Form 10-K", "United States", "Risk Factors",
    "Forward-Looking Statements", "Management", "Board of Directors"
}

def clip_text(s: str, n: int = 1000) -> str:
    s = (s or "").strip()
    return s if len(s) <= n else (s[:n] + "…")

def bundle_terms(terms: List[str], bundle_size: int = 4) -> List[str]:
    out = []
    for i in range(0, len(terms), bundle_size):
        out.append(" ".join(terms[i:i+bundle_size]))
    return [q for q in out if q.strip()]

def extract_doc_lexicon_terms(doc: DocumentYear, max_terms: int = 25, min_freq: int = 3) -> List[str]:
    """
    Deterministic company-year lexicon:
    - Acronyms (PBM, IoT, SaaS, etc.)
    - Capitalized phrases (MinuteClinic, Health Services, Ultium Platform, etc.)
    """
    text = "\n".join(p.text for p in doc.paragraphs)

    acronyms = re.findall(r"\b[A-Z]{2,8}\b", text)
    cap_phrases = re.findall(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2}\b", text)

    # Basic filtering
    def ok_phrase(x: str) -> bool:
        if len(x) < 3:
            return False
        if x in STOP_PHRASES:
            return False
        if x.lower() in {"the", "and", "for", "with"}:
            return False
        return True

    c = Counter([a for a in acronyms if ok_phrase(a)])
    c.update([p for p in cap_phrases if ok_phrase(p)])

    # Drop very generic tokens
    for bad in ["Company", "Companies", "Business", "Segment", "Segments", "Note", "Notes"]:
        c.pop(bad, None)

    terms = [t for t, n in c.most_common(max_terms * 2) if n >= min_freq]
    return terms[:max_terms]

def build_candidate_pool(idx: DocTfidfIndex, query_texts: List[str], k_per_query: int = 15) -> Dict[str, Tuple[str, float]]:
    """
    Returns pid -> (text, best_score_across_queries)
    """
    pool: Dict[str, Tuple[str, float]] = {}
    for q in query_texts:
        for pid, txt, score in idx.top_k(q, k=k_per_query):
            prev = pool.get(pid)
            if (prev is None) or (score > prev[1]):
                pool[pid] = (txt, score)
    return pool

def pool_to_ranked_ids(pool: Dict[str, Tuple[str, float]], max_paras: int) -> List[str]:
    ranked = sorted(pool.items(), key=lambda kv: kv[1][1], reverse=True)[:max_paras]
    return [pid for pid, _ in ranked]

def format_evidence_pack(doc: DocumentYear, para_ids: List[str], clip_chars: int = 1000, bracket_ids: bool = True) -> str:
    want = set(para_ids)
    selected = [p for p in doc.paragraphs if p.para_id in want]
    selected.sort(key=lambda p: p.para_id)
    if bracket_ids:
        return "\n\n".join([f"[{p.para_id}] {clip_text(p.text, clip_chars)}" for p in selected])
    return "\n\n".join([f"{p.para_id}: {clip_text(p.text, clip_chars)}" for p in selected])

# -------- Selector schema + agent --------

IDList6 = conlist(str, max_length=6)  # forces maxItems=6 in JSON schema

IDList6 = conlist(str, max_length=6)  # forces maxItems=6 in the JSON schema

class ParagraphSelection(BaseModel):
    bmcs_customer_segments: IDList6 = Field(default_factory=list)
    bmcs_offer: IDList6 = Field(default_factory=list)
    bmcs_channels: IDList6 = Field(default_factory=list)
    bmcs_revenue_model: IDList6 = Field(default_factory=list)
    bmcs_key_activities: IDList6 = Field(default_factory=list)
    reinvention_signals: IDList6 = Field(default_factory=list)

    type_xaaS: IDList6 = Field(default_factory=list)
    type_digital_products: IDList6 = Field(default_factory=list)
    type_physical_connected_products: IDList6 = Field(default_factory=list)
    type_ecosystem_platform: IDList6 = Field(default_factory=list)
    type_channel_innovation: IDList6 = Field(default_factory=list)

class ParagraphSelectorAgent:
    """
    Small selector: chooses paragraph IDs from a bounded candidate set.
    This is NOT doing final classification; it is improving recall.
    """
    name = "Selector"

    def __init__(self, llm: GeminiJSONClient, model: str):
        self.llm = llm
        self.model = model

    def build_prompt(self, doc: DocumentYear, candidates_pack: str) -> str:
        return f"""
Prompt Redacted

CANDIDATES (ONLY source of IDs):
{candidates_pack}

Return JSON per schema.
""".strip()

    def run(self, doc: DocumentYear, candidates_pack: str) -> ParagraphSelection:
        prompt = self.build_prompt(doc, candidates_pack)
        return self.llm.generate_structured(
            model=self.model,
            schema_model=ParagraphSelection,
            system_instruction="You only select IDs from candidates; never invent IDs.",
            prompt=prompt,
            temperature=0.0,
            max_output_tokens=20000,
            retries=4,
        )


In [ ]:
# -----------------------------
# Structured output schemas
# -----------------------------

Presence = Literal[0, 1, "0", "1", "?"]
Confidence = Literal["Low", "Medium", "High"]
Intensity = Literal["Low", "Medium", "High"]


class EvidenceText(BaseModel):
    text: str = Field(..., description="Concise summary text for this BMCS field.")
    evidence_para_ids: List[str] = Field(
        default_factory=list,
        description="Paragraph IDs from Item 1 supporting the summary. Must reference existing paragraph IDs.",
    )


class Initiative(BaseModel):
    initiative_id: str = Field(..., description="Must be chosen from CANONICAL_INITIATIVE_LABELS.")
    initiative_alias: Optional[str] = Field(
        default=None,
        description="REQUIRED if initiative_id='Other'. Short alias (<=6 words).",
    )
    presence: Presence = Field(..., description="1 if clearly present per evidence; 0 if clearly absent; '?' if uncertain.")
    evidence_para_ids: List[str] = Field(default_factory=list, description="Evidence paragraph IDs for this initiative.") 
    @field_validator("presence", mode="before")
    @classmethod
    def _coerce_presence(cls, v):
        # Accept common LLM variants and normalize
        if v is None:
            return "?"
        if isinstance(v, bool):
            return 1 if v else 0
        if isinstance(v, (int, float)) and v in (0, 1):
            return int(v)
        if isinstance(v, str):
            s = v.strip()
            if s in ("0", "1"):
                return int(s)
            if s in ("?", "unknown", "uncertain", "na", "n/a"):
                return "?"
        return v

    

class YearBMCS(BaseModel):
    ticker: str
    year: int

    # 9-field BMCS (adapt as needed)
    customer_segments: EvidenceText
    offer: EvidenceText
    channels: EvidenceText
    customer_relationships: EvidenceText
    revenue_model: EvidenceText
    key_resources: EvidenceText
    key_activities: EvidenceText
    key_partners: EvidenceText
    cost_structure: EvidenceText

    initiatives: List[Initiative] = Field(default_factory=list)

    notes: Optional[str] = Field(default=None, description="Optional caveats about uncertainty.")

def empty_bmcs(ticker: str, year: int) -> YearBMCS:
    """
    Safe fallback BMCS with all required fields populated as EvidenceText objects.
    Prevents Pydantic validation errors when generation fails.
    """
    # Helper to create empty evidence matching the EvidenceText schema
    def empty_ev(): 
        return EvidenceText(text="Not Disclosed (Generation Failed)", evidence_para_ids=[])

    return YearBMCS(
        ticker=str(ticker).upper(),
        year=year if year else 0,
        # Initialize all fields with proper objects, NOT strings
        customer_segments=empty_ev(),
        offer=empty_ev(),
        channels=empty_ev(),
        customer_relationships=empty_ev(),
        revenue_model=empty_ev(),
        key_resources=empty_ev(),
        key_activities=empty_ev(),
        key_partners=empty_ev(),
        cost_structure=empty_ev(),
        initiatives=[], 
        notes="Data unavailable due to generation error",
    )

class FieldAudit(BaseModel):
    field_name: str
    has_evidence: bool = Field(..., description="Does this field have >=1 evidence ID?")
    evidence_ids_valid: bool = Field(..., description="Do all IDs appear in the evidence pack?")
    supported_by_evidence: bool = Field(..., description="Does the text summary match the cited paragraph?")
    action: Literal["keep", "revise_summary", "remove_claim", "set_unknown"] = Field(..., description="Action to take.")
    comment: str = Field(..., description="Short explanation of the issue (if any).")

class CriticResult(BaseModel):
    approved: bool = Field(..., description="True ONLY if all field_checks pass with 'keep'.")
    field_checks: List[FieldAudit] = Field(..., description="Audit row for every BMCS field.")
    issues: List[str] = Field(default_factory=list, description="High-level summary of taxonomy or logic violations.")
    bmcs: YearBMCS


class PwCTypeJudgement(BaseModel):
    type_id: str = Field(..., description="One of: xaaS, digital_products, physical_connected_products, ecosystem_platform, channel_innovation")
    centrality: Literal[0, 1, 2] = Field(..., description="0=Absent, 1=Peripheral/Ambiguous, 2=Central/Core.")
    presence: Presence = Field(..., description="Derived from centrality: 0->0, 1->?, 2->1.")
    confidence: Confidence
    supporting_para_ids: List[str] = Field(default_factory=list)
    rationale: str = Field(..., description="Format: (i) PwC criterion matched, (ii) quote fragment, (iii) para_id.")

class TypeYearResult(BaseModel):
    ticker: str
    year: int
    judgements: List[PwCTypeJudgement]
    # Helpful aggregates (LLM can output; we validate + override deterministically later)
    type_count_present: int = 0
    type_count_unknown: int = 0
    bmri_binary: int = 0
    bmri_intensity_level: Intensity = "Low"
    notes: Optional[str] = None

# -----------------------------
# Chatbot Schema (Feedback & User Request)
# -----------------------------

class ChatbotAnswer(BaseModel):
    answer: str
    cites: List[str] = Field(
        default_factory=list, 
        description="Strict citations: <path>#<field> or <ticker>_<year>_<para_id>."
    )
    not_in_context: List[str] = Field(
        default_factory=list, 
        description="List of specific data points requested but missing from context."
    )   



In [ ]:
# -----------------------------
# LLM client (Gemini) + caching
# -----------------------------

class LLMError(RuntimeError):
    pass


@dataclass
class ModelConfig:
    selector: str = "gemini-2.5-flash-lite"
    bmcs_builder: str = "gemini-2.5-flash"
    bmcs_critic: str = "gemini-2.5-pro"
    type_rater: str = "gemini-2.5-pro"
    type_critic: str = "gemini-2.5-flash"
    writer: str = "gemini-3-pro-preview"
    editor: str = "gemini-2.5-pro"
    chatbot: str = "gemini-2.5-flash-lite"
    
def atomic_write_text(path: Path, text: str, encoding: str = "utf-8") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp.{os.getpid()}.{threading.get_ident()}")
    with open(tmp, "w", encoding=encoding) as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)  # atomic on POSIX + Windows

@contextmanager
def file_lock(lock_path: Path, timeout_s: float = 60.0, poll_s: float = 0.10):
    start = time.time()
    fd = None
    while True:
        try:
            fd = os.open(lock_path, os.O_CREAT | os.O_EXCL | os.O_RDWR)
            os.write(fd, str(os.getpid()).encode("utf-8"))
            break
        except FileExistsError:
            if time.time() - start > timeout_s:
                raise TimeoutError(f"Timed out waiting for lock: {lock_path}")
            time.sleep(poll_s + random.random() * 0.05)

    try:
        yield
    finally:
        try:
            if fd is not None:
                os.close(fd)
        finally:
            lock_path.unlink(missing_ok=True)

@contextmanager
def maybe_semaphore(sem: threading.Semaphore | None):
    if sem is None:
        yield
        return
    sem.acquire()
    try:
        yield
    finally:
        sem.release()


class GeminiJSONClient:
    """
    - Structured outputs with json_repair.loads
    - Disk cache with atomic write + per-key lock for thread safety
    - Safety filters disabled to prevent corporate data blocking
    - Thread-local genai.Client instances
    """
    CACHE_VERSION = "v4"  # Bumped to v4 to clear any 'double-encoded' corrupted files

    def __init__(
        self,
        api_key: str,
        cache_dir: Path,
        call_semaphore: threading.Semaphore | None = None,
        offline_cache_only: bool = False,
    ):
        self._api_key = api_key
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self._call_sem = call_semaphore
        self.offline_cache_only = offline_cache_only
        self._local = threading.local()

    def _get_client(self):
        c = getattr(self._local, "client", None)
        if c is None:
            c = genai.Client(api_key=self._api_key)
            self._local.client = c
        return c

    def _clean_json_text(self, text: str) -> str:
        """Helper for string-based cleaning if needed."""
        import json_repair
        try:
            return json_repair.repair_json(text)
        except Exception:
            return (text or "").strip()

    @staticmethod
    def _norm(s: str) -> str:
        return (s or "").replace("\r\n", "\n").strip()

    @classmethod
    def _hash_payload(
        cls, model, system, prompt, schema, temperature, max_output_tokens
    ) -> str:
        h = hashlib.sha256()
        h.update(cls.CACHE_VERSION.encode("utf-8"))
        h.update(model.encode("utf-8"))
        h.update(str(temperature).encode("utf-8"))
        h.update(str(max_output_tokens).encode("utf-8"))
        h.update(cls._norm(system).encode("utf-8"))
        h.update(cls._norm(prompt).encode("utf-8"))
        h.update(json.dumps(schema, sort_keys=True).encode("utf-8"))
        return h.hexdigest()

    def generate_structured(
        self,
        *,
        model: str,
        schema_model: type[BaseModel],
        system_instruction: str,
        prompt: str,
        temperature: float = 0.0,
        max_output_tokens: int = 16384,
        retries: int = 4,
    ) -> BaseModel:
        import json_repair
        schema = schema_model.model_json_schema()

        cache_key = self._hash_payload(
            model, system_instruction, prompt, schema, temperature, max_output_tokens
        )
        cache_path = self.cache_dir / f"{cache_key}.json"
        lock_path = self.cache_dir / f"{cache_key}.lock"

        # 1. Multi-thread Safe Cache Check
        with file_lock(lock_path):
            if cache_path.exists():
                try:
                    return schema_model.model_validate_json(cache_path.read_text(encoding="utf-8"))
                except Exception:
                    cache_path.unlink(missing_ok=True)

            if self.offline_cache_only:
                raise RuntimeError(f"Cache miss: {cache_path.name}")

            last_err: Optional[Exception] = None
            for attempt in range(retries + 1):
                try:
                    with maybe_semaphore(self._call_sem):
                        resp = self._get_client().models.generate_content(
                            model=model,
                            contents=prompt,
                            config={
                                "system_instruction": system_instruction,
                                "temperature": temperature,
                                "max_output_tokens": max_output_tokens,
                                "response_mime_type": "application/json",
                                "response_json_schema": schema,
                                # KEEP: Crucial safety settings from your original file
                                "safety_settings": [
                                    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
                                    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                                    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
                                    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                                ],
                            },
                        )

                    raw = resp.text or ""
                    
                    # 2. FIXED: Use loads() to get a Dict, then model_validate()
                    decoded_obj = json_repair.loads(raw)
                    obj = schema_model.model_validate(decoded_obj)

                    # 3. Store clean JSON string
                    atomic_write_text(cache_path, json.dumps(decoded_obj), encoding="utf-8")
                    return obj

                except Exception as e:
                    last_err = e
                    time.sleep((2 ** attempt) + random.random())

            raise LLMError(f"Gemini call failed after retries: {last_err}") from last_err

In [ ]:
def get_sector_specific_queries(sector: str) -> Dict[str, str]:
    """
    Returns tighter, industry-specific search queries to improve TF-IDF accuracy.
    """
    s = (sector or "").lower()
    
    # Defaults (Generic)
    q = {
        "customer_segments": "customers clients target market end-users demographic",
        "offer": "products services solutions platform portfolio offering",
        "channels": "sales marketing distribution direct dealers online",
        "revenue_model": "revenue sales fees income subscription pricing",
        "key_activities": "manufacturing research development operations production",
        "reinvention_signals": "transformation restructuring strategic shift pivot new business",
    }

    # 1. AUTOMOTIVE (GM, F, TSLA)
    if "auto" in s or "transport" in s:
        q["offer"] += " vehicles electric autonomous mobility parts accessories software-defined"
        q["revenue_model"] += " automotive sales leasing financing services regulatory credits"
        q["channels"] += " dealership network direct-to-consumer over-the-air updates"
        q["key_activities"] += " assembly supply chain battery production design engineering"

    # 2. FINANCIAL / BANKS (JPM, BAC, GS)
    elif "bank" in s or "finance" in s or "capital" in s:
        q["customer_segments"] += " consumer banking wealth management institutional commercial clients"
        q["offer"] += " deposits loans credit cards investment banking trading advisory assets"
        q["revenue_model"] += " net interest income non-interest fees commissions assets under management AUM"
        q["channels"] += " branches digital banking mobile app advisors"
        q["key_activities"] += " risk management underwriting liquidity capital allocation compliance"

    # 3. HEALTHCARE / PHARMA (PFE, JNJ, CVS)
    elif "health" in s or "pharma" in s or "bio" in s:
        q["customer_segments"] += " patients providers hospitals pharmacies payers governments"
        q["offer"] += " drugs therapeutics vaccines medical devices diagnostics care delivery"
        q["revenue_model"] += " product sales alliance revenue royalties premiums reimbursement"
        q["channels"] += " direct sales force wholesalers pharmacy benefit managers PBM"
        q["key_activities"] += " clinical trials R&D regulatory approval FDA manufacturing supply chain"

    # 4. TECH / SOFTWARE (MSFT, GOOGL, META)
    elif "tech" in s or "software" in s or "semicon" in s:
        q["offer"] += " cloud SaaS AI hardware operating system applications advertising"
        q["revenue_model"] += " subscription licensing advertising consumption-based recurring ARR"
        q["channels"] += " enterprise sales marketplace app store partners"
        q["key_activities"] += " software engineering data center operations innovation acquisition"

    # 5. RETAIL (WMT, TGT, AMZN)
    elif "retail" in s or "consumer" in s:
        q["customer_segments"] += " shoppers members subscribers loyalty program"
        q["offer"] += " merchandise private label groceries apparel electronics delivery"
        q["revenue_model"] += " net sales membership fees third-party seller services advertising"
        q["channels"] += " brick-and-mortar e-commerce omnichannel curbside pickup"
        q["key_activities"] += " logistics inventory management sourcing merchandising"

    return q

In [ ]:
# AI Agents 

class BMCSBuilderAgent:
    name = "BMCSBuilder"

    def __init__(self, llm: GeminiJSONClient, model: str, pwc_defs: Dict[str, str]):
        self.llm = llm
        self.model = model
        self.pwc_defs = pwc_defs

    def build_prompt(self, doc: DocumentYear, evidence_pack: str) -> str:
        # Dynamic ID example based on actual doc format
        example_id = doc.paragraphs[0].para_id if doc.paragraphs else "p0001"
        
        return f"""
Prompt Redacted

EVIDENCE PARAGRAPHS (Item 1):
{evidence_pack}
""".strip()

    def run(
    self,
    doc: DocumentYear,
    idx: DocTfidfIndex,
    top_k: int = 15,
    selection: ParagraphSelection | None = None,
    cfg: PipelineConfig | None = None,
) -> YearBMCS:

        # --- Keep: Explicit Map Tickers to Sectors ---
        sector_map = {
            # AUTOMOTIVE
            "GM": "Automotive", "F": "Automotive", "TSLA": "Automotive",
            "CMI": "Automotive", "PCAR": "Automotive", "AN": "Automotive",
            "BWA": "Automotive", "HOG": "Automotive", "PII": "Automotive", "GT": "Automotive",

            # FINANCE / BANKS
            "JPM": "Finance", "BAC": "Finance", "WFC": "Finance", "GS": "Finance",
            "MS": "Finance", "AXP": "Finance", "V": "Finance", "PYPL": "Finance",
            "COF": "Finance", "SCHW": "Finance",

            # HEALTHCARE
            "UNH": "Healthcare", "ELV": "Healthcare", "JNJ": "Healthcare",
            "PFE": "Healthcare", "MRK": "Healthcare", "ABT": "Healthcare",
            "TMO": "Healthcare", "LLY": "Healthcare", "AMGN": "Healthcare",
            "GILD": "Healthcare", "CVS": "Healthcare",

            # RETAIL
            "WMT": "Retail", "AMZN": "Retail", "COST": "Retail", "HD": "Retail",
            "KR": "Retail", "TGT": "Retail", "LOW": "Retail", "TJX": "Retail", "M": "Retail",

            # TECH
            "AAPL": "Tech", "MSFT": "Tech", "GOOGL": "Tech", "META": "Tech",
            "NVDA": "Tech", "ORCL": "Tech", "ADBE": "Tech", "CRM": "Tech",
            "CSCO": "Tech", "IBM": "Tech"
        }

        actual_sector = sector_map.get(doc.ticker, getattr(doc, "sector", "") or "")
        sector_queries = get_sector_specific_queries(actual_sector)

        # --- New: discovery queries to capture reinvention / structural shifts ---
        discovery_queries = [
            "Redacted"
        ]

        # --- New: dynamic company-year lexicon (captures firm-specific program names) ---
        lex_terms = extract_doc_lexicon_terms(doc, max_terms=25, min_freq=3)
        lex_queries = bundle_terms(lex_terms, bundle_size=4)

        # --- Build a ranked candidate pool across all query sources ---
        k_per_query = cfg.selector_k_per_query if cfg else top_k
        pool = build_candidate_pool(
            idx,
            list(sector_queries.values()) + discovery_queries + lex_queries,
            k_per_query=k_per_query
        )

        # Global cap on candidates (prevents huge evidence packs)
        max_candidates = cfg.selector_max_candidates if cfg else 120
        candidate_ids = pool_to_ranked_ids(pool, max_paras=max_candidates)

        # --- Optional: prioritize selector IDs (if provided) ---
        selector_ids: list[str] = []
        if selection is not None:
            selector_ids = (
                selection.bmcs_customer_segments
                + selection.bmcs_offer
                + selection.bmcs_channels
                + selection.bmcs_revenue_model
                + selection.bmcs_key_activities
                + selection.reinvention_signals
            )

        # Union selector IDs + candidates (stable order, dedupe)
        union_ids = list(dict.fromkeys(selector_ids + candidate_ids))

        # Final evidence cap + clipping
        max_paras = cfg.evidence_max_paras_bmcs if cfg else 35
        clip_chars = cfg.evidence_clip_chars if cfg else 1000
        union_ids = union_ids[:max_paras]

        evidence_pack = format_evidence_pack(
            doc,
            union_ids,
            clip_chars=clip_chars,
            bracket_ids=True
        )

        prompt = self.build_prompt(doc, evidence_pack)

        bmcs = self.llm.generate_structured(
            model=self.model,
            schema_model=YearBMCS,
            system_instruction=(
                "You are a strict, evidence-based analyst. "
                "Keep each field concise (1-3 sentences). Output ONLY valid JSON."
            ),
            prompt=prompt,
            temperature=0.0,
            max_output_tokens=20000,
            retries=4,
        )
        return bmcs


In [ ]:
class BMCSCriticAgent:
    name = "BMCSCritic"

    def __init__(self, llm: GeminiJSONClient, model: str):
        self.llm = llm
        self.model = model

    def build_prompt(self, bmcs: YearBMCS, evidence_pack: str) -> str:
        # FEEDBACK #2: Explicitly listing allowed IDs so the model knows what is valid
        # (This is handled implicitly by the evidence pack, but we reinforce it here)
        
        return f"""
Prompt Redacted

EVIDENCE PACK (The only valid source of truth):
{evidence_pack}

CURRENT BMCS JSON:
{bmcs.model_dump_json(exclude_none=True)}
""".strip()

    def run(self, bmcs: YearBMCS, evidence_pack: str) -> CriticResult:
        prompt = self.build_prompt(bmcs, evidence_pack)
        res = self.llm.generate_structured(
            model=self.model,
            schema_model=CriticResult,
            system_instruction="You are a strict compliance auditor who never hallucinates evidence.",
            prompt=prompt,
            temperature=0.0,
            max_output_tokens=10000,
            retries=4,
        )
        return res  # type: ignore[return-value]


In [ ]:
class TypeRaterAgent:
    name = "TypeRater"

    def __init__(self, llm: GeminiJSONClient, model: str, pwc_defs: Dict[str, str]):
        self.llm = llm
        self.model = model
        self.pwc_defs = pwc_defs

    # 1. UPDATED: Added 'sector_rules' argument to function signature
    def build_prompt(self, doc: DocumentYear, bmcs: YearBMCS, pwc_defs_text: str, evidence_pack: str, sector_rules: str) -> str:
        
        return f"""
You are a conservative classifier for PwC Business Model Reinvention (BMRI) types.
Classify {doc.ticker} ({doc.sector}) fiscal year {doc.year}.

INPUTS:
1. PwC DEFINITIONS (The Ground Truth):
{pwc_defs_text}

2. SECTOR-SPECIFIC DOMAIN RULES (CRITICAL):
{sector_rules}

3. BUSINESS MODEL CANVAS (Summary Context):
{bmcs.model_dump_json(exclude={'evidence_para_ids'})}

4. ITEM 1 EVIDENCE PARAGRAPHS (Source of Citations):
{evidence_pack}

RULES:
Prompt redacted
 
Return JSON per schema.
""".strip()

    def run(
        self,
        doc: DocumentYear,
        bmcs: YearBMCS,
        index: DocTfidfIndex,
        top_k_per_type: int = 12,
        selection: ParagraphSelection | None = None,
    ) -> TypeYearResult:

        # --- 2. NEW: Define Sector Rules Here ---
        SECTOR_DOMAIN_RULES = {
            "Sector_Name": "Prompt Redacted"
        }
        DEFAULT_SECTOR_RULE = "Focus on distinguishing between core legacy operations and genuine business model reinvention signals."

        # --- 3. NEW: Logic to Select the Correct Rule ---
        current_sector_rules = DEFAULT_SECTOR_RULE
        if doc.sector:
            # Simple substring match (e.g. "Healthcare" matches "Health Care")
            for key, rules in SECTOR_DOMAIN_RULES.items():
                if key.lower() in doc.sector.lower():
                    current_sector_rules = rules
                    break
        
        # ---------------------------------------------------------

        queries = {
            "BM_DENSE": "business model overview segments strategy revenue products services customers",
            "xaaS": "subscription recurring revenue membership fee service contract managed services",
            "digital_products": "software platform digital products apps cloud data analytics AI",
            "physical_connected_products": "connected devices IoT sensors remote monitoring embedded software",
            "ecosystem_platform": "platform ecosystem marketplace partners network multi-party system",
            "channel_innovation": "direct-to-consumer omnichannel distribution retail footprint online channels",
        }

        discovery_q = [
            "acquisition integration divestiture segment reorganization restructuring",
            "partnership alliance ecosystem platform network",
            "pricing model monetization recurring revenue subscription",
        ]

        lex_terms = extract_doc_lexicon_terms(doc, max_terms=25, min_freq=3)
        lex_q = bundle_terms(lex_terms, bundle_size=4)

        pool = build_candidate_pool(index, list(queries.values()) + discovery_q + lex_q, k_per_query=top_k_per_type)
        candidate_ids = pool_to_ranked_ids(pool, max_paras=120)

        selected_ids: List[str] = []
        if selection is not None:
            selected_ids = (
                selection.type_xaaS
                + selection.type_digital_products
                + selection.type_physical_connected_products
                + selection.type_ecosystem_platform
                + selection.type_channel_innovation
            )

        union_ids = list(dict.fromkeys(selected_ids + candidate_ids))
        union_ids = union_ids[:35]
        evidence_pack = format_evidence_pack(doc, union_ids, clip_chars=1000, bracket_ids=False)

        pwc_defs_text = self._format_pwc_defs()

        # 4. UPDATED: Pass 'current_sector_rules' to build_prompt
        prompt = self.build_prompt(doc, bmcs, pwc_defs_text, evidence_pack, current_sector_rules)
        
        res = self.llm.generate_structured(
            model=self.model,
            schema_model=TypeYearResult,
            system_instruction="You are a conservative classifier. Keep rationale short. If unsure, choose Score 1 (?).",
            prompt=prompt,
            temperature=0.0,
            max_output_tokens=60000,  
            retries=4,
        )

        return res

    def _format_pwc_defs(self) -> str:
        blocks = []
        if "general" in self.pwc_defs:
            blocks.append("GENERAL CONCEPT:\n" + self.pwc_defs["general"])
        for pretty, tid in PWC_TYPE_IDS.items():
            if tid in self.pwc_defs:
                blocks.append(f"\n--- DEFINITION FOR: {pretty} ({tid}) ---\n{self.pwc_defs[tid]}")
        return "\n\n".join(blocks).strip()

In [ ]:
class TypeCriticAgent:
    name = "TypeCritic"

    def __init__(self, llm: GeminiJSONClient, model: str):
        self.llm = llm
        self.model = model

    def build_prompt(self, tyr: TypeYearResult, evidence_pack: str) -> str:
        return f"""
Prompt Redacted

EVIDENCE PACK:
{evidence_pack}

PROPOSED JUDGEMENTS:
{tyr.model_dump_json()}

Return the corrected TypeYearResult.
""".strip()

    def run(self, tyr: TypeYearResult, evidence_pack: str) -> TypeYearResult:
        prompt = self.build_prompt(tyr, evidence_pack)
        res = self.llm.generate_structured(
            model=self.model,
            schema_model=TypeYearResult,
            system_instruction="You are a strict auditor. You downgrade claims that lack proof.",
            prompt=prompt,
            temperature=0.0,
            max_output_tokens=60000,
            retries=4,

        )
        return res # type: ignore[return-value]

In [ ]:
class ChangeNarratorAgent:
    """
    Optional LLM agent that writes a forensic 'diff' of changes between years.
    Enforces evidence citations and context-aware gaps.
    """
    name = "ChangeNarrator"

    def __init__(self, llm: GeminiJSONClient, model: str):
        self.llm = llm
        self.model = model

    class _ChangeNote(BaseModel):
        ticker: str
        year: int
        prev_year: Optional[int]
        gap_years: Optional[int]
        material_change: int
        
        # New Structured Fields (Feedback A)
        high_level_changes: List[str] = Field(..., description="Bullet points of what explicitly changed (max 6).")
        stability_notes: List[str] = Field(..., description="Bullet points of what stayed the same.")
        supporting_para_ids: List[str] = Field(..., description="List of evidence paragraph IDs cited from the BMCS evidence.")
        caution_notes: Optional[List[str]] = Field(default=None, description="Notes on uncertainty or missing years.")

    def build_prompt(
        self, 
        ticker: str, 
        year: int, 
        prev_year: Optional[int], 
        gap_years: Optional[int], 
        distance_percentile: float, # New Input
        bmcs_now: YearBMCS, 
        bmcs_prev: Optional[YearBMCS], 
        material_change: int
    ) -> str:
        
        # Formatting context for the prompt
        gap_context = f"This transition covers a {gap_years}-year gap." if (gap_years and gap_years > 1) else "This is a consecutive year-over-year comparison."
        
        material_context = (
            f"Statistically, this year is a MATERIAL CHANGE (Top {int((1.0-distance_percentile)*100)}% of firm's history)." 
            if material_change else 
            f"Statistically, this year is STABLE (Bottom {int(distance_percentile*100)}% of firm's history)."
        )

        return f"""
You are an ANALYST comparing Business Model Canvas Summaries (BMCS) for {ticker}: {prev_year} -> {year}.

CONTEXT:
1. {gap_context}
2. {material_context}

RULES (Feedback B & C):
Prompt Redacted

BMCS (Previous - {prev_year}):
{bmcs_prev.model_dump_json() if bmcs_prev else "N/A (First year or missing data)"}

BMCS (Current - {year}):
{bmcs_now.model_dump_json()}

Return structured JSON per schema.
""".strip()

    def run(
        self, 
        *, 
        ticker: str, 
        year: int, 
        prev_year: Optional[int], 
        gap_years: Optional[int], 
        distance_percentile: float, 
        bmcs_now: YearBMCS, 
        bmcs_prev: Optional[YearBMCS], 
        material_change: int
    ) -> "_ChangeNote":
        
        prompt = self.build_prompt(
            ticker, year, prev_year, gap_years, distance_percentile, 
            bmcs_now, bmcs_prev, material_change
        )
        
        note = self.llm.generate_structured(
            model=self.model,
            schema_model=self._ChangeNote,
            system_instruction="You write strict, evidence-based change logs.",
            prompt=prompt,
            temperature=0.2, # Kept at 0.2 as requested
            max_output_tokens=2048,
        )
        
        # Deterministic override of metadata
        note.ticker = ticker
        note.year = year
        note.prev_year = prev_year
        note.gap_years = gap_years
        note.material_change = material_change
        
        return note

In [ ]:
class WriterAgent:
    name = "Writer"

    def __init__(self, llm: GeminiJSONClient, model: str):
        self.llm = llm
        self.model = model

    def build_prompt(self, ticker: str, panel_csv: str, pivots_csv: str) -> str:
        return f"""
Prompt Redacted
"""

    def run(self, ticker: str, panel: pd.DataFrame, pivots: pd.DataFrame) -> str:
        # Convert dataframes to CSV string for the prompt
        panel_csv = panel.to_csv(index=False)
        pivots_csv = pivots.to_csv(index=False)
        
        prompt = self.build_prompt(ticker, panel_csv, pivots_csv)

        # Using the Writer Model (Gemini 1.5 Pro or 3)
        # We use generate_content (raw text) because the output is Markdown, not JSON
        resp = self.llm._get_client().models.generate_content(
            model=self.model,
            contents=prompt,
            config={
                "temperature": 0.2, # Low temp for analytical precision
                "max_output_tokens": 8192,
                "system_instruction": "You are a specialized financial analyst focusing on business model transformation.",
            }
        )
        
        return resp.text

In [ ]:
class EditorAgent:
    """Optional agent that edits the WriterAgent markdown for clarity and thesis tone."""
    name = "Editor"

    class _Edit(BaseModel):
        ticker: str
        markdown: str
        edits: List[Dict[str, str]] = Field(
            ..., 
            description="List of edits made. Each dict must have: type, location, before, after."
        )
        potential_fact_risks: List[str] = Field(
            default_factory=list, 
            description="List of any statements that seemed borderline or lacked strong CSV support."
        )
    def __init__(self, llm: GeminiJSONClient, model: str):
        self.llm = llm
        self.model = model

    def build_prompt(self, ticker: str, draft_markdown: str) -> str:
        return f"""
Prompt Redacted

TICKER: {ticker}

DRAFT MARKDOWN:
{draft_markdown}

Return JSON with the edited markdown.
""".strip()

    def run(self, ticker: str, draft_markdown: str) -> Dict[str, Any]:
        edited = self.llm.generate_structured(
            model=self.model,
            schema_model=self._Edit,
            system_instruction="You are a strict editor who preserves factual constraints.",
            prompt=self.build_prompt(ticker, draft_markdown),
            temperature=0.2,
            max_output_tokens=8192,
        )
        # --- THE FIX: Return Dict, not String ---
        return edited.model_dump()

In [ ]:
class ChatbotAgent:
    """
    Interactive Q&A over already-produced outputs.
    Answers must be grounded only in the provided context pack.
    """
    name = "Chatbot"

    def __init__(self, llm: GeminiJSONClient, model: str):
        self.llm = llm
        self.model = model

    def build_prompt(self, question: str, context_pack: str) -> str:
        # FEEDBACK: Incorporating strict grounding and citation formats
        return f"""
You are a CLOSED-BOOK thesis assistant. 

GROUNDING RULES (MANDATORY):
- Use ONLY the CONTEXT PACK below. Do NOT use outside knowledge.
- If the answer is not supported by the context, say "Not answerable from provided context" and list missing items in the 'not_in_context' field.
- Do NOT use outside knowledge or general assumptions.

CITATION RULES:
- Every claim must have a citation.
- JSON format: "<path>#<field>" (e.g., "AAPL/bmcs_2019.json#offer")
- CSV format: "<path>:<row_selector>" (e.g., "AAPL/panel.csv:year=2019")
- Paragraph format: "<ticker>_<year>_<para_id>"

ANSWER STYLE:
- Be concise. Use bullet points for evidence.
- Do not infer causality or performance success unless explicitly stated in context.

QUESTION:
{question}

CONTEXT PACK:
{context_pack}

Return JSON with answer, cites, and any missing context items.
""".strip()

    def answer(self, question: str, context_pack: str) -> Dict[str, Any]:
        # Uses temperature=0.2 as requested by user
        res = self.llm.generate_structured(
            model=self.model,
            schema_model=ChatbotAnswer, # Matches the schema name above
            system_instruction="You are a strict, evidence-only research assistant.",
            prompt=self.build_prompt(question, context_pack),
            temperature=0.2, 
            max_output_tokens=2048,
        )
        # Return the dictionary so citations aren't lost (Feedback #1)
        return res.model_dump()

In [ ]:
# -----------------------------
# Deterministic validations & repairs
# -----------------------------

def validate_evidence_ids_exist(doc: DocumentYear, ids: Iterable[str]) -> List[str]:
    invalid = [i for i in ids if i not in doc.para_id_set]
    return invalid


def bmcs_collect_all_evidence_ids(bmcs: YearBMCS) -> Dict[str, List[str]]:
    """
    Returns mapping field_name -> evidence ids list.
    """
    m: Dict[str, List[str]] = {}
    for field in (
        "customer_segments",
        "offer",
        "channels",
        "customer_relationships",
        "revenue_model",
        "key_resources",
        "key_activities",
        "key_partners",
        "cost_structure",
    ):
        ev = getattr(bmcs, field).evidence_para_ids
        m[field] = list(ev or [])
    # initiatives
    for i, init in enumerate(bmcs.initiatives):
        m[f"initiatives[{i}].evidence_para_ids"] = list(init.evidence_para_ids or [])
    return m


def enforce_evidence_nonempty(bmcs: YearBMCS, required_fields: Sequence[str]) -> List[str]:
    missing = []
    for f in required_fields:
        ev = getattr(bmcs, f).evidence_para_ids
        if not ev or len(ev) < 1:
            missing.append(f)
    return missing


def validate_initiative_ids(bmcs: YearBMCS) -> List[str]:
    bad = []
    allowed = set(CANONICAL_INITIATIVE_LABELS)
    for i, init in enumerate(bmcs.initiatives):
        if init.initiative_id not in allowed:
            bad.append(f"initiatives[{i}].initiative_id={init.initiative_id}")
    return bad


def validate_type_supporting_ids(doc: DocumentYear, tyr: TypeYearResult) -> List[str]:
    bad: List[str] = []
    for j, item in enumerate(tyr.judgements):
        invalid = validate_evidence_ids_exist(doc, item.supporting_para_ids)
        if invalid:
            bad.append(f"judgements[{j}].supporting_para_ids invalid: {invalid}")
    return bad


def apply_type_aggregates_deterministically(doc: DocumentYear, tyr: TypeYearResult, material_change: bool) -> TypeYearResult:
    """
    Overwrites aggregate fields deterministically + applies intensity guardrail.

    Guardrail:
    - Unknowns ('?') treated as 0 for summary counts.
    - If any type is '?' and it would otherwise produce High intensity,
      cap at Medium unless there's at least one confident presence=1 plus material_change.
    """
    present = sum(1 for j in tyr.judgements if j.presence == 1)
    unknown = sum(1 for j in tyr.judgements if j.presence == "?")

    tyr.type_count_present = int(present)
    tyr.type_count_unknown = int(unknown)
    tyr.bmri_binary = 1 if present > 0 else 0

    # equal weights across 5 types
    raw = present / 5.0
    if raw > (2/3):
        level: Intensity = "High"
    elif raw >= (1/3):
        level = "Medium"
    else:
        level = "Low"

    # Guardrail against High from weak signals
    if level == "High" and unknown > 0:
        any_confident_present = any(
            (j.presence == 1 and j.confidence in ("Medium", "High")) for j in tyr.judgements
        )
        if not (any_confident_present and material_change):
            level = "Medium"

    tyr.bmri_intensity_level = level
    return tyr


def smooth_type_results(type_results_by_year: Dict[int, TypeYearResult]) -> Dict[int, TypeYearResult]:
    """
    Fills 1-year gaps. If a type is Present (1) in Year T-1 and Year T+1, 
    but Absent/Unknown (0/?) in Year T, force Year T to Present (1).
    """
    years = sorted(type_results_by_year.keys())
    # Identify all type_ids from the first available year
    if not years: return type_results_by_year
    
    all_types = {j.type_id for j in type_results_by_year[years[0]].judgements}
    
    for t_id in all_types:
        for i in range(1, len(years) - 1):
            prev_y, curr_y, next_y = years[i-1], years[i], years[i+1]
            
            # Get judgements for this specific type (SAFE lookups)
            prev_j = next((j for j in type_results_by_year[prev_y].judgements if j.type_id == t_id), None)
            curr_j = next((j for j in type_results_by_year[curr_y].judgements if j.type_id == t_id), None)
            next_j = next((j for j in type_results_by_year[next_y].judgements if j.type_id == t_id), None)

            # If current doesn't exist, we can't smooth/update it
            if curr_j is None:
                continue

            # If either neighbor is missing, skip smoothing for this type_id
            if prev_j is None or next_j is None:
                continue

            # GAP LOGIC: 1 -> (0 or ?) -> 1  ==> FIX TO 1
            if prev_j.presence == 1 and next_j.presence == 1 and curr_j.presence != 1:
                print(f"SMOOTHING: Filled gap for {t_id} in {curr_y} (Context: {prev_y}=1, {next_y}=1)")
                curr_j.presence = 1
                curr_j.centrality = 2
                curr_j.rationale += " [AUTO-SMOOTHED due to continuity]"
                
    return type_results_by_year

In [ ]:
# -----------------------------
# Deterministic transitions (presence/onset/offset/pivots)
# -----------------------------

@dataclass
class TransitionRow:
    ticker: str
    year: int
    prev_year: Optional[int]
    gap_years: Optional[int]
    type_id: str
    presence: Presence
    prev_presence: Optional[Presence]
    onset: int
    offset: int
    transition_unknown: int
    exit_cue: int


def compute_exit_cue(doc: DocumentYear, para_ids: Sequence[str]) -> int:
    text = " ".join([p.text for p in doc.paragraphs if p.para_id in set(para_ids)])
    t = text.lower()
    for pat in EXIT_CUE_PATTERNS:
        if re.search(pat, t):
            return 1
    return 0


def compute_transitions_for_firm(
    docs_by_year: Dict[int, DocumentYear],
    type_results_by_year: Dict[int, TypeYearResult],
) -> List[TransitionRow]:
    years = sorted(type_results_by_year.keys())
    out: List[TransitionRow] = []

    for idx, y in enumerate(years):
        tyr = type_results_by_year[y]
        prev_y = years[idx - 1] if idx > 0 else None

        gap = (y - prev_y) if prev_y is not None else None

        prev_tyr = type_results_by_year.get(prev_y) if prev_y is not None else None

        prev_map: Dict[str, Presence] = {}
        if prev_tyr is not None:
            prev_map = {j.type_id: j.presence for j in prev_tyr.judgements}

        for j in tyr.judgements:
            prev_presence: Optional[Presence] = prev_map.get(j.type_id) if prev_y is not None else None

            # Known-state requirement:
            # Onset=1 only if presence(t)=1 AND presence(t-1)=0
            # Offset=1 only if presence(t)=0 AND presence(t-1)=1
            # If either side is '?', onset/offset=0 and transition_unknown=1
            onset = 0
            offset = 0
            transition_unknown = 0
            if prev_presence is None:
                # no previous year
                pass
            else:
                if j.presence == "?" or prev_presence == "?":
                    transition_unknown = 1
                else:
                    if j.presence == 1 and prev_presence == 0:
                        onset = 1
                    if j.presence == 0 and prev_presence == 1:
                        offset = 1

            doc = docs_by_year[y]
            exit_cue = compute_exit_cue(doc, j.supporting_para_ids) if offset == 1 else 0

            out.append(
                TransitionRow(
                    ticker=tyr.ticker,
                    year=y,
                    prev_year=prev_y,
                    gap_years=gap,
                    type_id=j.type_id,
                    presence=j.presence,
                    prev_presence=prev_presence,
                    onset=onset,
                    offset=offset,
                    transition_unknown=transition_unknown,
                    exit_cue=exit_cue,
                )
            )

    return out


@dataclass
class PivotRow:
    ticker: str
    year: int
    prev_year: Optional[int]
    gap_years: Optional[int]
    material_change: int
    pivot: int
    exit_pivot: int


def compute_pivots(
    transitions: List[TransitionRow],
    material_change_years: Dict[int, int],
) -> List[PivotRow]:
    """
    Pivot rules (as requested):
    Default:
      Pivot(t) = any onset AND material_change(t)
    Secondary:
      ExitPivot(t) = any offset AND material_change(t) AND exit_cue=1
    """
    by_year: Dict[int, List[TransitionRow]] = {}
    for tr in transitions:
        by_year.setdefault(tr.year, []).append(tr)

    out: List[PivotRow] = []
    for year, rows in sorted(by_year.items()):
        mat = int(material_change_years.get(year, 0))
        any_onset = any(r.onset == 1 for r in rows)
        any_exit_offset = any((r.offset == 1 and r.exit_cue == 1) for r in rows)

        pivot = 1 if (any_onset and mat == 1) else 0
        exit_pivot = 1 if (any_exit_offset and mat == 1) else 0

        # Prev_year/gap_years are same for all types within year (use first)
        prev_year = rows[0].prev_year
        gap = rows[0].gap_years
        out.append(PivotRow(
            ticker=rows[0].ticker,
            year=year,
            prev_year=prev_year,
            gap_years=gap,
            material_change=mat,
            pivot=pivot,
            exit_pivot=exit_pivot,
        ))
    return out


In [ ]:
# -----------------------------
# Material change detection (within-firm percentile)
# -----------------------------

def bmcs_snapshot_text(bmcs: YearBMCS) -> str:
    """
    Creates a compact textual representation of BMCS for similarity comparisons.
    Evidence IDs are excluded by design; we compare *content summaries*.
    """
    parts = [
        bmcs.customer_segments.text,
        bmcs.offer.text,
        bmcs.channels.text,
        bmcs.customer_relationships.text,
        bmcs.revenue_model.text,
        bmcs.key_resources.text,
        bmcs.key_activities.text,
        bmcs.key_partners.text,
        bmcs.cost_structure.text,
    ]
    return " | ".join([p.strip() for p in parts if p and p.strip()])


def compute_material_change_within_firm(
    bmcs_by_year: Dict[int, YearBMCS],
    top_percent: float = 0.20,
) -> Tuple[Dict[int, int], pd.DataFrame]:
    """
    For each firm:
      - Fit TF-IDF on all BMCS snapshot texts for that firm (stable vector space within firm)
      - Compute cosine distances for consecutive available years
      - Rank distances within firm
      - Material change = distance in top X% of that firm's own changes

    Returns:
      - material_change_years: year -> 0/1 for the *current* year in each transition
      - df_distances: rows per transition with distance + rank/percentile
    """
    years = sorted(bmcs_by_year.keys())
    if len(years) < 2:
        return {y: 0 for y in years}, pd.DataFrame()

    texts = [bmcs_snapshot_text(bmcs_by_year[y]) for y in years]

    vect = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b")
    X = vect.fit_transform(texts)

    # consecutive transitions (prev available year)
    distances = []
    transitions = []
    for i in range(1, len(years)):
        y_prev = years[i - 1]
        y = years[i]
        sim = cosine_similarity(X[i], X[i - 1])[0, 0]
        dist = 1.0 - float(sim)
        distances.append(dist)
        transitions.append((y_prev, y, y - y_prev))

    # rank distances (higher dist = more change)
    dist_arr = np.array(distances)
    order = dist_arr.argsort()  # ascending
    ranks = np.empty_like(order)
    ranks[order] = np.arange(len(dist_arr))
    percentiles = ranks / max(1, (len(dist_arr) - 1))

    # Determine threshold by percentile (top X%)
    # Example: top 20% => percentile >= 0.80
    cutoff = 1.0 - top_percent
    material_flags = (percentiles >= cutoff).astype(int)

    material_change_years: Dict[int, int] = {years[0]: 0}
    rows = []
    for i, (y_prev, y, gap) in enumerate(transitions):
        material_change_years[y] = int(material_flags[i])
        rows.append({
            "prev_year": y_prev,
            "year": y,
            "gap_years": gap,
            "distance": float(dist_arr[i]),
            "rank": int(ranks[i]),
            "percentile": float(percentiles[i]),
            "material_change": int(material_flags[i]),
        })

    df = pd.DataFrame(rows)
    return material_change_years, df


In [ ]:
# -----------------------------
# Finance join
# -----------------------------

def load_financial_data(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path)
    df = df.rename(columns={
        "(tic) Ticker Symbol": "ticker",
        "(fyear) Data Year - Fiscal": "year",
        "NAT ": "nat",
        "BMRI_abs_delta_NAT": "bmri_abs_delta_nat",
        "BMRI_pace3": "bmri_pace3",
        "Sector": "sector",
    })
    df["ticker"] = df["ticker"].astype(str).str.upper()
    df["year"] = df["year"].astype(int)
    return df


# -----------------------------
# Orchestrator (Planner)
# -----------------------------

@dataclass
class PipelineConfig:
    data_dir: Path
    output_dir: Path
    pwc_types_rtf: Path
    tickers_rtf: Optional[Path] = None
    financial_xlsx: Optional[Path] = None

    year_min: int = 2014
    year_max: int = 2024

    retrieval_top_k_bmcs: int = 5
    retrieval_top_k_types: int = 6

    material_change_top_percent: float = 0.20  # default top 20%
    material_change_sensitivity: Tuple[float, float] = (0.10, 0.30)
    
    # Selector (recall booster)
    enable_selector: bool = True
    selector_max_candidates: int = 120
    selector_k_per_query: int = 15
    evidence_max_paras_bmcs: int = 35
    evidence_max_paras_types: int = 35
    evidence_clip_chars: int = 1000


    # LLM models by agent
    models: ModelConfig = dataclasses.field(default_factory=ModelConfig)

    # Evidence enforcement
    evidence_fields: Tuple[str, ...] = DEFAULT_EVIDENCE_FIELDS

    # Execution toggles
    use_critic: bool = True
    use_type_rater: bool = True
    dry_run_no_llm: bool = False

    # Optional writing/chatbot stages (extra API spend)
    make_report: bool = True
    enable_chatbot: bool = False
    
    # Parallelism (speed + rate-limit control)
    max_workers_companies: int = 3
    max_workers_years: int = 4
    max_concurrent_llm_calls: int = 8

    # Caching control
    offline_cache_only: bool = False
    cache_dir: Optional[Path] = None


def save_evidence_audit(company_dir: Path, ticker: str, bmcs_by_year: Dict[int, YearBMCS], docs_by_year: Dict[int, DocumentYear]):
    """
    Creates a master audit file mapping every claim -> evidence IDs -> raw text.
    Saves as an Excel (.xlsx) file.
    """
    audit_rows = []

    for year, bmcs in bmcs_by_year.items():
        doc = docs_by_year.get(year)
        if not doc: continue
        
        # 1. Map Paragraph IDs to Text for this year
        para_map = {p.para_id: p.text for p in doc.paragraphs}

        # 2. Extract Evidence for every BMCS field
        fields_to_audit = [
            "customer_segments", "offer", "channels", "revenue_model", 
            "key_resources", "key_activities"
        ]
        
        for field in fields_to_audit:
            item = getattr(bmcs, field)
            cited_ids = item.evidence_para_ids
            
            # Retrieve the actual text for those IDs
            evidence_text = "\n".join([f"[{pid}] {para_map.get(pid, 'ERR: ID not found')}" for pid in cited_ids])
            
            audit_rows.append({
                "Ticker": ticker,
                "Year": year,
                "Category": f"BMCS_{field}",
                "Summary_Claim": item.text,
                "Evidence_IDs": ", ".join(cited_ids),
                "Raw_Source_Text": evidence_text 
            })

    # Save to EXCEL
    if audit_rows:
        df_audit = pd.DataFrame(audit_rows)
        # Use the existing save_xlsx helper from your code
        # Note the filename change to .xlsx
        audit_path = company_dir / f"{ticker}_EVIDENCE_AUDIT.xlsx"
        save_xlsx(df_audit, audit_path, sheet_name="Audit_Trail")
        print(f"Saved audit trail for {ticker} (Excel)") 


In [ ]:
class BMRIPlannerAgent:
    name = "Planner"

    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        self.output_dir = cfg.output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.pwc_defs = parse_pwc_definitions_from_rtf(cfg.pwc_types_rtf)

        self.ticker_name_map: Dict[str, str] = {}
        if cfg.tickers_rtf and cfg.tickers_rtf.exists():
            self.ticker_name_map = parse_ticker_company_pairs_from_rtf(cfg.tickers_rtf)

        self.fin_df: Optional[pd.DataFrame] = None
        if cfg.financial_xlsx and cfg.financial_xlsx.exists():
            self.fin_df = load_financial_data(cfg.financial_xlsx)

        self.llm: Optional[GeminiJSONClient] = None
        self._llm_semaphore: Optional[threading.Semaphore] = None

        if not cfg.dry_run_no_llm:
            self._llm_semaphore = threading.BoundedSemaphore(cfg.max_concurrent_llm_calls)
            api_key = load_api_key()
            cache_dir = cfg.cache_dir or (self.output_dir / "_llm_cache")
            self.llm = GeminiJSONClient(
                api_key=api_key,
                cache_dir=cache_dir,
                call_semaphore=self._llm_semaphore,
                offline_cache_only=cfg.offline_cache_only,
            )

        self.selector = ParagraphSelectorAgent(self.llm, cfg.models.selector) if (self.llm and cfg.enable_selector) else None

    def _log_status(self, msg: str):
        print(f"[{time.strftime('%H:%M:%S')}] {msg}")

    def run(self, tickers: Sequence[str]) -> None:
        tickers = [t.upper() for t in tickers]
        with ThreadPoolExecutor(max_workers=self.cfg.max_workers_companies) as ex:
            futs = {ex.submit(self.run_one_company, t): t for t in tickers}
            for fut in tqdm(as_completed(futs), total=len(tickers), desc="Overall Progress"):
                t = futs[fut]
                try:
                    fut.result()
                except Exception:
                    print(f"\n[ERROR] Company {t} failed (full traceback):")
                    traceback.print_exc()
                    print("\n" + "-"*80 + "\n")

    def run_one_company(self, ticker: str) -> None:
        self._log_status(f"=== Starting {ticker} ===")
        company_dir = self.output_dir / ticker
        company_dir.mkdir(parents=True, exist_ok=True)

        docs = self._load_docs_for_ticker(ticker)
        if not docs:
            print(f"[WARN] No docs found for {ticker}")
            return

        docs_by_year = {d.year: d for d in docs}
        indexes = {y: DocTfidfIndex(docs_by_year[y].paragraphs) for y in sorted(docs_by_year)}
        selection_by_year: Dict[int, Optional[ParagraphSelection]] = {}

        def get_selection_for_year(year: int):
            if self.selector is None: return None
            if year in selection_by_year: return selection_by_year[year]
            
            doc, idx = docs_by_year[year], indexes[year]
            self._log_status(f"Selector -> {ticker} {year}")
            
            base_q = list(get_sector_specific_queries(doc.sector or "").values())
            pool = build_candidate_pool(idx, base_q, k_per_query=self.cfg.selector_k_per_query)
            candidate_ids = pool_to_ranked_ids(pool, max_paras=60)
            candidates_pack = format_evidence_pack(doc, candidate_ids, clip_chars=450, bracket_ids=True)

            try:
                sel = self.selector.run(doc, candidates_pack)
                selection_by_year[year] = sel
                return sel
            except Exception as e:
                print(f"[WARN] Selector failed for {ticker} {year}: {e}")
                selection_by_year[year] = None
                return None

        # STAGE 2: BMCS Building
        bmcs_by_year: Dict[int, YearBMCS] = {}
        
        def build_bmcs_for_year(year: int):
            """
            Worker function with internal error handling.
            Returns valid empty_bmcs on failure so the pipeline keeps moving.
            """
            try:
                self._log_status(f"BMCS Builder -> {ticker} {year}")
                sel = get_selection_for_year(year)
                # Attempt generation
                bmcs = self._stage2_build_bmcs(docs_by_year[year], indexes[year], selection=sel)
                return year, bmcs
                
            except Exception as e:
                # Log the specific error but do not crash the thread
                print(f"\n[ERROR] BMCS generation failed for {ticker} {year}: {e}")
                # Return the safe fallback so the rest of the pipeline works
                return year, empty_bmcs(ticker, year)

        # Execution loop (now safe from individual year failures)
        years = sorted(docs_by_year.keys())
        with ThreadPoolExecutor(max_workers=self.cfg.max_workers_years) as ex:
            futs = [ex.submit(build_bmcs_for_year, y) for y in years]
            
            # We can now iterate safely because exceptions are caught inside the future
            for fut in tqdm(as_completed(futs), total=len(years), desc=f"BMCS: {ticker}", leave=False):
                y, bmcs = fut.result() # This will now always return a tuple, never raise
                bmcs_by_year[y] = bmcs

        # STAGE 4: Material Change
        material_flags, df_dist = compute_material_change_within_firm(bmcs_by_year, self.cfg.material_change_top_percent)
        write_outputs_for_company(company_dir, "bmcs_change_distances", df_dist)
        
        # --- NEW: SENSITIVITY ANALYSES (10% and 30% files) ---
        for pct in [0.10, 0.30]:
            _, df_sens = compute_material_change_within_firm(bmcs_by_year, top_percent=pct)
            write_outputs_for_company(company_dir, f"bmcs_change_distances_top{int(pct*100)}pct", df_sens)

        # STAGE 5: Type Rating
        type_results_by_year: Dict[int, TypeYearResult] = {}
        if self.cfg.use_type_rater:
            def rate_types_for_year(year: int):
                self._log_status(f"Type Rater -> {ticker} {year}")
                sel = get_selection_for_year(year) 
                bmcs_for_year = bmcs_by_year[year]
                
                # PASS IT TO THE FUNCTION
                bmcs = bmcs_by_year.get(year)
                if bmcs is None:
                    print(f"[WARN] Missing BMCS for {ticker} {year}; using empty BMCS fallback.")
                    bmcs = empty_bmcs(ticker, year)


                tyr = self._stage5_rate_types(docs_by_year[year], bmcs, indexes[year], selection=sel)
                
                # Apply deterministic aggregates
                tyr = apply_type_aggregates_deterministically(
                    docs_by_year[year], tyr, bool(material_flags.get(year, 0))
                )
                return year, tyr

            with ThreadPoolExecutor(max_workers=self.cfg.max_workers_years) as ex:
                futs = [ex.submit(rate_types_for_year, y) for y in years]
                for fut in tqdm(as_completed(futs), total=len(years), desc=f"Rating: {ticker}", leave=False):
                    y, tyr = fut.result()
                    type_results_by_year[y] = tyr

        type_results_by_year = smooth_type_results(type_results_by_year)

        # STAGE 6: Transitions & Final Panel
        transitions = compute_transitions_for_firm(docs_by_year, type_results_by_year)
        pivots = compute_pivots(transitions, material_flags)
        panel = self._build_panel(company_dir, ticker, bmcs_by_year, type_results_by_year, material_flags)
        
        save_xlsx(panel, company_dir / "panel.xlsx", sheet_name="panel")
        save_xlsx(pd.DataFrame([dataclasses.asdict(t) for t in transitions]), company_dir / "transitions.xlsx")
        save_xlsx(pd.DataFrame([dataclasses.asdict(p) for p in pivots]), company_dir / "pivots.xlsx")
        # This uses the data currently in memory (re-loaded from cache)
        save_evidence_audit(company_dir, ticker, bmcs_by_year, docs_by_year)

        # --- STAGE 7: REPORTING (NEW) ---
        if self.cfg.make_report and self.llm:
            self._log_status(f"Writer Agent -> Generating Report for {ticker}")
            writer = WriterAgent(self.llm, self.cfg.models.writer)
            editor = EditorAgent(self.llm, self.cfg.models.editor)
            
            pivots_df = pd.DataFrame([dataclasses.asdict(p) for p in pivots])
            
            # Generate draft report
            draft_md = writer.run(ticker, panel, pivots_df)

            # [FIX] Save the unedited draft to disk
            atomic_write_text(company_dir / f"{ticker}_draft_report.md", draft_md)
            
            # Run Editor for final polish and changelog
            edited_res = editor.run(ticker, draft_md)
            
            # Save final report and editor changelog
            atomic_write_text(company_dir / f"{ticker}_final_report.md", edited_res['markdown'])
            atomic_write_text(company_dir / "editor_changelog.json", json.dumps(edited_res['edits'], indent=2))
            self._log_status(f"Report and editor changelog saved for {ticker}")

        self._log_status(f"[OK] Completed {ticker}")

    # --- INTERNAL STAGE METHODS ---

    def _stage2_build_bmcs(self, doc: DocumentYear, idx: DocTfidfIndex, selection: Optional[ParagraphSelection] = None) -> YearBMCS:
        builder = BMCSBuilderAgent(self.llm, self.cfg.models.bmcs_builder, self.pwc_defs)
        bmcs = builder.run(doc, idx, top_k=self.cfg.retrieval_top_k_bmcs, selection=selection, cfg=self.cfg)
        bmcs = self._repair_bmcs_until_valid(doc, idx, bmcs)

        if self.cfg.use_critic:
            critic = BMCSCriticAgent(self.llm, self.cfg.models.bmcs_critic)
            try:
                referenced_ids = set()
                for ids in bmcs_collect_all_evidence_ids(bmcs).values(): referenced_ids.update(ids)
                pack = "\n\n".join(f"{p.para_id}: {p.text}" for p in doc.paragraphs if p.para_id in referenced_ids)
                crit_res = critic.run(bmcs, pack)
                if crit_res.bmcs: bmcs = crit_res.bmcs
            except Exception as e:
                print(f"[WARN] BMCS critic failed for {doc.ticker} {doc.year}: {e}")
        return self._repair_bmcs_until_valid(doc, idx, bmcs)

    def _stage5_rate_types(
        self,
        doc: DocumentYear,
        bmcs: YearBMCS,
        idx: DocTfidfIndex,
        selection: Optional[ParagraphSelection] = None
    ) -> TypeYearResult:
        rater = TypeRaterAgent(self.llm, self.cfg.models.type_rater, self.pwc_defs)
        tyr = rater.run(
            doc,
            bmcs,
            idx,
            top_k_per_type=self.cfg.retrieval_top_k_types,
            selection=selection
        )

        if self.cfg.use_critic:
            critic = TypeCriticAgent(self.llm, self.cfg.models.type_critic)
            try:
                referenced_ids = {pid for j in tyr.judgements for pid in j.supporting_para_ids}
                pack = "\n\n".join(f"{p.para_id}: {p.text}" for p in doc.paragraphs if p.para_id in referenced_ids)
                tyr = critic.run(tyr, pack)
            except Exception as e:
                print(f"[WARN] Type critic failed for {doc.ticker} {doc.year}: {e}")
        return tyr

    def _repair_bmcs_until_valid(self, doc: DocumentYear, idx: DocTfidfIndex, bmcs: YearBMCS, max_rounds: int = 1) -> YearBMCS:
        # Simplified repair logic from your original code
        for _ in range(max_rounds):
            ev_map = bmcs_collect_all_evidence_ids(bmcs)
            invalid = {f: ids for f, ids in ev_map.items() if validate_evidence_ids_exist(doc, ids)}
            if not invalid: break
            # Prompt logic would go here if round > 0
        return bmcs

    def _load_docs_for_ticker(self, ticker: str) -> List[DocumentYear]:
        docs = []
        for p in self.cfg.data_dir.glob("*.txt"):
            meta = parse_item1_filename(p)
            if meta and meta.ticker.upper() == ticker.upper():
                raw = p.read_text(errors="ignore")
                paras = split_into_paragraphs(clean_text(raw))
                docs.append(DocumentYear(ticker=ticker, year=meta.year, sector=meta.sector, company=meta.company, path=p, paragraphs=paras))
        docs.sort(key=lambda d: d.year)
        return docs

    def _build_panel(self, company_dir: Path, ticker: str, bmcs_by_year: Dict[int, YearBMCS], type_results_by_year: Dict[int, TypeYearResult], material_flags: Dict[int, int]) -> pd.DataFrame:
        rows = []
        for y in sorted(bmcs_by_year.keys()):
            bmcs, tyr = bmcs_by_year[y], type_results_by_year.get(y)
            
            # 1. AI-Generated Data
            row = {
                "ticker": ticker, 
                "year": y, 
                "material_change": material_flags.get(y, 0), 
                "bmcs_offer": bmcs.offer.text
            }
            
            if tyr:
                row.update({"bmri_binary": tyr.bmri_binary, "intensity": tyr.bmri_intensity_level})
                for j in tyr.judgements: 
                    row[f"type_{j.type_id}"] = 1 if j.presence == 1 else 0

            # 2. Financial Data Merge (The Fix)
            rows.append(row)
        df_panel = pd.DataFrame(rows)

        # ---- FINANCE MERGE (NEW) ----
        if self.fin_df is not None:
            fin_cols = [c for c in self.fin_df.columns if c not in {"sector"}]

            df_panel = df_panel.merge(
                self.fin_df[fin_cols],
                on=["ticker", "year"],
                how="left",
                validate="1:1"
            )

        return df_panel


In [ ]:
# -----------------------------
# API key loader
# -----------------------------

def load_api_key() -> str:
    """
    Loads API key from .env (preferred) or environment.

    User provides: MY_API_KEY in .env.
    google-genai also auto-reads GEMINI_API_KEY or GOOGLE_API_KEY if set 
    """
    from dotenv import load_dotenv
    import os
    
    load_dotenv()

    key = os.getenv("MY_API_KEY") or os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if not key:
        raise RuntimeError("No API key found. Set MY_API_KEY in .env (or GEMINI_API_KEY / GOOGLE_API_KEY).")
    return key

In [ ]:
# -----------------------------
# Example entrypoint
# -----------------------------

def run_test():
    """
    Minimal test run on the sample files in a folder.
    Edit tickers list to run 1-2 companies first.
    """
    data_dir = Path("/Users/Admin/Downloads/Thesis/thesis_dataset correct years")  # <-- change to your folder with Item 1 .txt files
    out_dir = Path("outputs")

    cfg = PipelineConfig(
        data_dir=data_dir,
        output_dir=out_dir,
        pwc_types_rtf=Path("/Users/Admin/Downloads/Thesis/PWC 5 types definition.rtf"),
        tickers_rtf=Path("/Users/Admin/Downloads/Thesis/Ticker & Company Names.rtf"),
        financial_xlsx=Path("/Users/Admin/Downloads/Thesis/Financial Data_NAT.xlsx"),
        use_critic=True,
        use_type_rater=True,
        dry_run_no_llm=False,
    )

    planner = BMRIPlannerAgent(cfg)

    tickers = [
        "Redacted",  # <-- add your tickers here (as they appear in the filenames and RTF)
    ]

    print(f"Starting analysis for {len(tickers)} companies...")
    planner.run(tickers)
    print("Run complete.")

run_test()
